In [13]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

In [14]:
odf = pd.read_csv("microStudy_withLogprobs.csv")
text_scores = pd.read_csv("raw_coh_scores.csv")

odf.head()
# text_scores.head()


,Unnamed: 0,key,text,charm_article_id,IR,bucket,guidewords,likes,dislikes,attestation,input_text,logprob,normalized_everything
0,0,FH_main,NORFOLK — Likening the provisions of the open ...,1,Inspired,+,"['righteous', 'pre-ordained', 'beautiful']",I am most deeply moved by things that feel rig...,NaN,I am most deeply moved by things that feel rig...,AUTHOR BIO: \n\nI am most deeply moved by thin...,-2.390625,-0.247495
1,1,FH_main,NORFOLK — Likening the provisions of the open ...,1,Inspired,-,"['false', 'uncreative', 'dull']",NaN,I have a visceral reaction to anything that st...,I have a visceral reaction to anything that st...,AUTHOR BIO: \n\nI have a visceral reaction to ...,-2.388672,-0.233510
2,2,FH_main,NORFOLK — Likening the provisions of the open ...,1,Popular,+,"['preferred', 'popular', 'favorite']",I take comfort in knowing what’s preferred by ...,NaN,I take comfort in knowing what’s preferred by ...,AUTHOR BIO: \n\nI take comfort in knowing what...,-2.386719,-0.219524
3,3,FH_main,NORFOLK — Likening the provisions of the open ...,1,Popular,-,"['resented', 'feared', 'isolated']",NaN,What unsettles me most is when something or so...,What unsettles me most is when something or so...,AUTHOR BIO: \n\nWhat unsettles me most is when...,-2.380859,-0.177567
4,4,FH_main,NORFOLK — Likening the provisions of the open ...,1,Moral,+,"['solidary', 'responsible', 'just']",I’m most at ease when I feel like people are l...,NaN,I’m most at ease when I feel like people are l...,AUTHOR BIO: \n\nI’m most at ease when I feel l...,-2.263672,0.661569


In [15]:
#first we can work with a smaller subset

pdf = odf[['key', 'IR', 'bucket', 'logprob', 'text_logprob']].copy()

KeyError: "['text_logprob'] not in index"

In [ ]:
#Method 1

In [17]:
#normalization is going to be a big part of this working well probably. right now we need to combine each ir into one score per article. 
#lets try normalization across everything:


mean_all = pdf['logprob'].mean()
std_all = pdf['logprob'].std()
pdf['normalized_everything'] = (pdf['logprob'] - mean_all) / std_all 


#for this, we get -0.24, -0.23 for the first set. roughly the same. however we can see the moral ir for this bucket has a 0.66 
#so it's working right there's just not a lot of coherence with the inspired frame for or against. 
#this means it should score close to 0, sum of the two values times the bucket seems promising?

pdf['meth1_score'] = pdf.apply(lambda row : row['normalized_everything'] if row['bucket'] == "+" else -row['normalized_everything'], axis = 1)
# pdf = pdf.sort_values(['key', 'IR', 'bucket'])
def agg_IRandKey(series):
    return series.sum()

# spdf = (
#     pdf.groupby(["key", "IR"], as_index=False)
#       .agg({"meth1_score": agg_IRandKey, 
#             "logprob" : list})
# )

spdf = (
    pdf.groupby(["key", "IR"], sort=False)
       .apply(lambda g: pd.Series({
           "meth1_score": agg_IRandKey(g["meth1_score"]),
           "logprob": g.sort_values("bucket", ascending=False)["meth1_score"].tolist()
       }))
       .reset_index()
)

spdf.head()

/scratch/slurm-6228092/ipykernel_1875350/3515413544.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,key,IR,meth1_score,logprob
0,FH_main,Inspired,-0.013986,"[0.23350980267912325, -0.24749541652942358]"
1,FH_main,Popular,-0.041957,"[0.17756734727792184, -0.2195241888288229]"
2,FH_main,Moral,0.377612,"[-0.28395790978198965, 0.661569483740099]"
3,FH_main,Civic,-0.797180,"[-0.5217133452370956, -0.2754666442300243]"
4,FH_main,Economic,0.097899,"[0.23350980267912325, -0.1356105057270208]"


In [18]:
# spdf[(spdf['key'] == 'FH_main') & (spdf['IR'] == 'Inspired')].head(20)
# pdf[(pdf['key'] == 'FH_main') & (pdf['IR'] == 'Inspired')].head(20)
#goal: -0.247495, 0.233510
# pdf.head()
# -0.247495 + 0.233510 = -0.013984999999999997

In [19]:
#so now we can compare these vectors. 
grouped_by_key = spdf[['key', 'meth1_score']].groupby(["key"], as_index = False).agg({"meth1_score" : list})
calt = np.array(grouped_by_key.loc[0, 'meth1_score'])
cm = np.array(grouped_by_key.loc[1, 'meth1_score'])
fhalt = np.array(grouped_by_key.loc[2, 'meth1_score'])
fhm = np.array(grouped_by_key.loc[3, 'meth1_score'])

cosim_cakes = np.dot(calt, cm) / (np.linalg.norm(calt) * np.linalg.norm(cm))
print(f"cosim between cakes is {cosim_cakes}")
cosim_fh = np.dot(fhalt, fhm) / (np.linalg.norm(fhalt) * np.linalg.norm(fhm))
print(f"cosim between fh is {cosim_fh}")
cosim_m = np.dot(cm, fhm) / (np.linalg.norm(cm) * np.linalg.norm(fhm))
print(f"cosim between m is {cosim_m}")
cosim_alt = np.dot(calt, fhalt) / (np.linalg.norm(calt) * np.linalg.norm(fhalt))
print(f"cosim between alt is {cosim_alt}")


#cosim between cakes is 0.9886576469656523
#cosim between fh is 0.9984421981272852
#cosim between m is 0.994393821496789
#cosim between alt is 0.9956894849923223
#uh oh


cosim between cakes is 0.9886576469656523
cosim between fh is 0.998442198127285
cosim between m is 0.9943938214967891
cosim between alt is 0.9956894849923223


In [20]:
#lets try diff distance metrics first. 
def wrapper_for_distances(df, scorecolname,  foo):
    grouped_by_key = df[['key', scorecolname]].groupby(["key"], as_index = False).agg({scorecolname : list})
    calt = np.array(grouped_by_key.loc[0, scorecolname])
    cm = np.array(grouped_by_key.loc[1, scorecolname])
    fhalt = np.array(grouped_by_key.loc[2, scorecolname])
    fhm = np.array(grouped_by_key.loc[3, scorecolname])

    cosim_cakes = foo(calt, cm)
    cosim_fh = foo(fhalt, fhm)
    cosim_m = foo(cm, fhm)
    cosim_alt = foo(calt, fhalt)
    
    all_nums = {
        "cakes" : cosim_cakes,
        "fh" : cosim_fh,
        "m" : cosim_m,
        "alt" : cosim_alt
    }

    print(f"cakes are {cosim_cakes}, which is greater than {[x for x in all_nums if all_nums[x] < cosim_cakes]}")
    print(f"fh are {cosim_fh}, which is greater than {[x for x in all_nums if all_nums[x] < cosim_fh]}")
    print(f"m are {cosim_m}, which is greater than {[x for x in all_nums if all_nums[x] < cosim_m]}")
    print(f"alt are {cosim_alt}, which is greater than {[x for x in all_nums if all_nums[x] < cosim_alt]}")
    
    
def euclidan(a, b):
    pairwise_diff = a-b
    sqd = pairwise_diff**2
    sumd = sqd.sum()
    sqrt = np.sqrt(sumd)
    return sqrt


wrapper_for_distances(spdf, 'meth1_score', euclidan)

cakes are 0.437818965868678, which is greater than []
fh are 0.8066923406099966, which is greater than ['cakes', 'm', 'alt']
m are 0.5887262381887962, which is greater than ['cakes', 'alt']
alt are 0.587562253525993, which is greater than ['cakes']


In [21]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
wrapper_for_distances(spdf, 'meth1_score', cosine)

cakes are 0.9886576469656523, which is greater than []
fh are 0.998442198127285, which is greater than ['cakes', 'm', 'alt']
m are 0.9943938214967891, which is greater than ['cakes']
alt are 0.9956894849923223, which is greater than ['cakes', 'm']


In [22]:

def pearson(a, b):
    corr_value, pval = pearsonr(a, b)
    return corr_value
wrapper_for_distances(spdf, 'meth1_score', pearson)

cakes are 0.9987785634040821, which is greater than ['fh', 'm']
fh are 0.998649119592657, which is greater than ['m']
m are 0.9977653778913241, which is greater than []
alt are 0.9989503983093013, which is greater than ['cakes', 'fh', 'm']


In [23]:
def spearman(a, b):
    corr_value, pval = spearmanr(a,b)
    return corr_value
wrapper_for_distances(spdf, 'meth1_score', spearman)

cakes are 0.9727272727272728, which is greater than ['fh', 'm']
fh are 0.9642857142857145, which is greater than ['m']
m are 0.9369749612033815, which is greater than []
alt are 0.991031208965115, which is greater than ['cakes', 'fh', 'm']


In [ ]:
fhm

In [ ]:
cm

In [ ]:
fhalt

In [ ]:
calt

## Method 2

In [24]:
#we could ofc try normalizing differently - i.e. across each IR instead of everything
stats = pdf[['IR', 'logprob']].groupby("IR")["logprob"].agg(
    ir_mean="mean", 
    ir_std="std"
).reset_index()

stats['IR'] = stats['IR'].astype(str) #UGHH
pdf['IR'] = pdf['IR'].astype(str)

if "ir_mean" not in list(pdf.columns):
    pdf = pdf.merge(stats, left_on="IR", right_on = "IR", how="left")
pdf["ir_norm_logprob"] = (pdf["logprob"] - pdf["ir_mean"]) / pdf["ir_std"] 
pdf['meth2_score'] = pdf.apply(lambda row : row['ir_norm_logprob'] if row['bucket'] == "+" else -row['ir_norm_logprob'], axis = 1)

def agg_IRandKey(series):
    return series.sum()


b2 = (
    pdf.groupby(["key", "IR"], sort=False)
       .apply(lambda g: pd.Series({
           "meth2_score": agg_IRandKey(g["meth2_score"]),
           "logprob": g.sort_values("bucket", ascending=False)["meth2_score"].tolist()
       }))
       .reset_index()
)
b2.head()
# pdf.head()
# stats.head()

/scratch/slurm-6228092/ipykernel_1875350/2611232464.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,key,IR,meth2_score,logprob
0,FH_main,Inspired,-0.015617,"[0.18740851426632738, -0.20302589045518799]"
1,FH_main,Popular,-0.050335,"[0.12164336580112224, -0.17197855164986248]"
2,FH_main,Moral,0.451093,"[0.5179218875327178, -0.06682863064938294]"
3,FH_main,Civic,-0.756322,"[-0.21395956175212544, -0.5423626100228296]"
4,FH_main,Economic,0.108264,"[0.212661358705396, -0.10439739427355804]"


In [25]:
wrapper_for_distances(b2, 'meth2_score', euclidan)
wrapper_for_distances(b2, 'meth2_score', cosine)

cakes are 0.41740454120038484, which is greater than []
fh are 0.7332384494136004, which is greater than ['cakes', 'm', 'alt']
m are 0.5352993994339738, which is greater than ['cakes']
alt are 0.5359231800367039, which is greater than ['cakes', 'm']
cakes are 0.9848568350974372, which is greater than []
fh are 0.9975906135883257, which is greater than ['cakes', 'm', 'alt']
m are 0.9926112387081343, which is greater than ['cakes']
alt are 0.9935269163210451, which is greater than ['cakes', 'm']


In [39]:
#try pearson?
from scipy.stats import pearsonr


def pearson(a, b):
    corr_value, pval = pearsonr(a, b)
    return corr_value
wrapper_for_distances(b2, 'meth2_score', pearson)

def pearson_dist(a, b):
    return 1 - pearson(a, b)

cakes are 0.9982986071284322, which is greater than ['fh', 'm', 'alt']
fh are 0.997909152671044, which is greater than ['m']
m are 0.9969231514389252, which is greater than []
alt are 0.9981833140005594, which is greater than ['fh', 'm']


In [27]:
#so yeah distance should be lower if closer, and m is the farthest away, which is quite cool.
#might change with more data?
#also we do need to control for similarity


def spearman(a, b):
    corr_value, pval = spearmanr(a,b)
    return corr_value
wrapper_for_distances(b2, 'meth2_score', spearman)

cakes are 1.0, which is greater than ['fh', 'm']
fh are 0.9642857142857145, which is greater than []
m are 0.9642857142857145, which is greater than []
alt are 1.0, which is greater than ['fh', 'm']


In [ ]:
#pearson seems to be perfect, as we want 1-correlation to get distance. 

## Method 3

In [ ]:
#thinking that maybe similarity is an issue here

In [34]:
#first we can work with a smaller subset
odf = pd.read_csv("microStudy_withLogprobs.csv")
text_scores = pd.read_csv("raw_coh_scores.csv")
text_scores['text_logprob'] = text_scores['avg_logprob']
odf = odf.merge(text_scores[['input_text', 'text_logprob']], on='input_text')

odf.head()
# text_scores.head()

pdf = odf[['key', 'IR', 'bucket', 'logprob', 'text_logprob']].copy()
pdf['norm_text'] = pdf['logprob'] - pdf['text_logprob']

#thinking i dont need to do this since already "normalizing"
# mean_all = pdf['norm_text'].mean()
# std_all = pdf['norm_text'].std()
# pdf['norm_all'] = (pdf['norm_text'] - mean_all) / std_all 
pdf['norm_all'] = pdf['norm_text']

pdf['meth3_score'] = pdf.apply(lambda row : row['norm_all'] if row['bucket'] == "+" else -row['norm_all'], axis = 1)
b3 = (
    pdf.groupby(["key", "IR"], sort=False)
       .apply(lambda g: pd.Series({
           "meth3_score": agg_IRandKey(g["meth3_score"]),
           "logprob": g.sort_values("bucket", ascending=False)["meth3_score"].tolist()
       }))
       .reset_index()
)

b3.head()

/scratch/slurm-6228092/ipykernel_1875350/1748561510.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,key,IR,meth3_score,logprob
0,FH_main,Inspired,-0.001953,"[-0.046875, 0.044921875]"
1,FH_main,Popular,-0.005859,"[-0.0546875, 0.048828125]"
2,FH_main,Moral,0.052734,"[-0.119140625, 0.171875]"
3,FH_main,Civic,-0.111328,"[-0.15234375, 0.041015625]"
4,FH_main,Economic,0.013672,"[-0.046875, 0.060546875]"


In [32]:
pdf.head()


,key,IR,bucket,logprob,text_logprob,norm_text,norm_all,meth3_score
0,FH_main,Inspired,+,-2.390625,-2.435547,0.044922,-0.200376,-0.200376
1,FH_main,Inspired,-,-2.388672,-2.435547,0.046875,-0.188724,0.188724
2,FH_main,Popular,+,-2.386719,-2.435547,0.048828,-0.177072,-0.177072
3,FH_main,Popular,-,-2.380859,-2.435547,0.054688,-0.142115,0.142115
4,FH_main,Moral,+,-2.263672,-2.435547,0.171875,0.557016,0.557016


In [40]:
print("pearson")
wrapper_for_distances(b3, 'meth3_score', pearson)
print("cosine")
wrapper_for_distances(b3, 'meth3_score', cosine)
print("euc")
wrapper_for_distances(b3, 'meth3_score', euclidan)
print("spearman")
wrapper_for_distances(b3, 'meth3_score', spearman)

pearson
cakes are 0.9987785634040822, which is greater than ['fh', 'm']
fh are 0.9986491195926571, which is greater than ['m']
m are 0.9977653778913242, which is greater than []
alt are 0.9989503983093013, which is greater than ['cakes', 'fh', 'm']
cosine
cakes are 0.9886576469656523, which is greater than []
fh are 0.998442198127285, which is greater than ['cakes', 'm', 'alt']
m are 0.994393821496789, which is greater than ['cakes']
alt are 0.9956894849923221, which is greater than ['cakes', 'm']
euc
cakes are 0.06114248375975988, which is greater than []
fh are 0.1126565479798417, which is greater than ['cakes', 'm', 'alt']
m are 0.08221705148378589, which is greater than ['cakes', 'alt']
alt are 0.08205449819375002, which is greater than ['cakes']
spearman
cakes are 0.9727272727272728, which is greater than ['fh', 'm']
fh are 0.9642857142857145, which is greater than ['m']
m are 0.9369749612033815, which is greater than []
alt are 0.991031208965115, which is greater than ['cakes', '